[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CharlesShang/TorchCode/blob/master/solutions/64_qlora_nf4_quantization_solution.ipynb)

# 🔴 Solution: QLoRA NF4 Quantization

Reference solution for `qlora_nf4_quantization`.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch


In [ ]:
# ✅ SOLUTION

class NF4Quantizer:
    def __init__(self, block_size: int = 64):
        self.block_size = block_size
        self.codebook = torch.tensor([
            -1.0000, -0.6962, -0.5251, -0.3949,
            -0.2844, -0.1848, -0.0911,  0.0000,
             0.0796,  0.1609,  0.2461,  0.3379,
             0.4407,  0.5626,  0.7230,  1.0000,
        ])

    def quantize(self, weight: torch.Tensor):
        original_shape = tuple(weight.shape)
        flat = weight.flatten()
        pad = (-flat.numel()) % self.block_size
        if pad:
            flat = torch.cat([flat, flat.new_zeros(pad)])
        blocks = flat.view(-1, self.block_size)
        scales = blocks.abs().amax(dim=1).clamp_min(1e-8)
        normalized = (blocks / scales[:, None]).clamp(-1, 1)
        codebook = self.codebook.to(device=weight.device, dtype=weight.dtype)
        distances = (normalized.unsqueeze(-1) - codebook.view(1, 1, 16)).abs()
        codes = distances.argmin(dim=-1).to(torch.uint8)
        return codes, scales, original_shape

    def dequantize(self, codes: torch.Tensor, scales: torch.Tensor, original_shape: tuple):
        codebook = self.codebook.to(device=scales.device, dtype=scales.dtype)
        values = codebook[codes.long()] * scales[:, None]
        return values.flatten()[:int(torch.tensor(original_shape).prod().item())].view(original_shape)


In [ ]:
# Verify
q = NF4Quantizer(block_size=8)
w = torch.randn(3, 5)
codes, scales, shape = q.quantize(w)
recon = q.dequantize(codes, scales, shape)
print(codes.shape, scales.shape, recon.shape, (w - recon).abs().mean().item())


In [ ]:
# Run judge
from torch_judge import check
check('qlora_nf4_quantization')
